In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print

/home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])

14:53:10 [info     ] Indexed datasets path: /home/gpudual/bhklab/josh/med-image_index/notebooks/indexed_datasets [imgnet] call=store.__init__:57


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [28]:
description = """
Despite the considerable progress in automatic abdominal multi-organ segmentation from CT/MRI scans in recent years, a comprehensive evaluation of the models' capabilities is hampered by the lack of a large-scale benchmark from diverse clinical scenarios. Constraint by the high cost of collecting and labeling 3D medical data, most of the deep learning models to date are driven by datasets with a limited number of organs of interest or samples, which still limits the power of modern deep models and makes it difficult to provide a fully comprehensive and fair estimate of various methods. To mitigate the limitations, we present AMOS, a large-scale, diverse, clinical dataset for abdominal organ segmentation. <u>AMOS provides 500 CT and 100 MRI scans collected from multi-center, multi-vendor, multi-modality, multi-phase, multi-disease patients, each with voxel-level annotations of 15 abdominal organs, providing challenging examples and test-bed for studying robust segmentation algorithms</u> under diverse targets and scenarios. We further benchmark several state-of-the-art medical segmentation models to evaluate the status of the existing methods on this new challenging dataset. We have made our datasets, benchmark servers, and baselines publicly available, and hope to inspire future research.  For more details, please refer to our paper "https://arxiv.org/pdf/2206.08023.pdf" as well as homepage "https://jiyuanfeng.github.io/AMOS/".

AMOS provides the following content. imagesTr and labelsTr provide 240 scans (200 CT and 40 MRI), imagesVa and labelsVa provide 120 scans for model selection (100 CT and 20 MRI), and imagesTs provide 120 test data (please submit your predictions from https://amos22.grand-challenge.org/evaluation/challenge/submissions to get a score). Please note that id numbers less than 500 belong to CT data, otherwise they belong to MRI data.
"""

source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="7155725",
    description=description,
)


In [29]:
dataset_index_path = (indexed_datasets.imgtools_path / "AMOS")
dataset_index_path.mkdir(parents=True, exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [33]:
print(indexed_datasets.source_config("AMOS"))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='zenodo',
    record_id='7155725',
    filenames=None,
    post_download=['unzip'],
    description='Despite the considerable progress in automatic abdominal multi-organ segmentation from CT/MRI 
scans in recent years, a comprehensive evaluation of the models\' capabilities is hampered by the lack of a 
large-scale benchmark from diverse clinical scenarios. Constraint by the high cost of collecting and labeling 3D 
medical data, most of the deep learning models to date are driven by datasets with a limited number of organs of 
interest or samples, which still limits the power of modern deep models and makes it difficult to provide a fully 
comprehensive and fair estimate of various methods. To mitigate the limitations, we present AMOS, a large-scale, 
diverse, clinical dataset for abdominal organ segmentation. <u>AMOS provides 500 CT and 100 MRI scans collected 
from multi-center, multi-vendor, multi-modality, multi-phase, multi-disease patients, each with voxel-level 
annotations of 15 abdominal organs, providing challenging examples and test-bed for studying robust segmentation 
algorithms</u> under diverse targets and scenarios. We further benchmark several state-of-the-art medical 
segmentation models to evaluate the status of the existing methods on this new challenging dataset. We have made 
our datasets, benchmark servers, and baselines publicly available, and hope to inspire future research.  For more 
details, please refer to our paper "https://arxiv.org/pdf/2206.08023.pdf" as well as homepage 
"https://jiyuanfeng.github.io/AMOS/".\n\nAMOS provides the following content. imagesTr and labelsTr provide 240 
scans (200 CT and 40 MRI), imagesVa and labelsVa provide 120 scans for model selection (100 CT and 20 MRI), and 
imagesTs provide 120 test data (please submit your predictions from 
https://amos22.grand-challenge.org/evaluation/challenge/submissions to get a score). Please note that id numbers 
less than 500 belong to CT data, otherwise they belong to MRI data.\n'
)

In [ ]:
temp_data_path = Path("temp_data") / "AMOS"
temp_data_path.mkdir(parents=True, exist_ok=True)
downloader = indexed_datasets.downloader("AMOS").download(temp_data_path)

amos22.zip:   1%|          | 285M/24.2G [01:20<1:37:13, 4.11MB/s] 

In [11]:
import imgtools.loggers
import logging

imgtools.loggers.get_logger("imgtools").setLevel(logging.INFO)

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="amos22/images{Split}/amos_{ImageID:d}.nii.gz",
    mask_name_pattern="amos22/labels{Split}/amos_{ImageID:d}.nii.gz",
    output_dir=indexed_datasets.imgtools_path,
    n_jobs=10,
    force=True,
    deep=True,
)
crawler.crawl()

12:28:26 [info     ] Starting NIFTI crawl.          [imgtools] nifti_dir=PosixPath('temp_data/AMOS') output_dir=PosixPath('indexed_datasets/.imgtools') dataset_name=None call=crawler.crawl:78
         [info     ] Found 1191 NIfTI files in /home/gpudual/bhklab/josh/med-image_index/notebooks/temp_data/AMOS. [imgtools] call=parse_niftis.parse_nifti_dir:426
         [info     ] Using shared keys: ['Split', 'ImageID'] for reference_id, this will be used to link masks to their referenced scans [imgtools] call=parse_niftis.parse_nifti_dir:447
Parsing 1191 files:  78%|███████▊  | 932/1191 [24:16<36:40,  8.49s/it]  /home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
14:12:23 [info     ] Parsing all NIfTI files took 6236.9513 seconds [imgtools] call=time

In [17]:
index_df = indexed_datasets.index("AMOS")

In [18]:
index_df

,Split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,nvoxels,...,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality
0,Tr,1,amos22/imagesTr/amos_0001.nii.gz,scan,Tr_1,MedImage,7afee8a38d53124559a13a6781f122286d3430f7,"(768, 768, 90)",3,53084160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
1,Tr,4,amos22/imagesTr/amos_0004.nii.gz,scan,Tr_4,MedImage,c30d47e4286bcdfd0de0e7a91f2c83fdbf5f6c73,"(512, 512, 78)",3,20447232,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
2,Tr,5,amos22/imagesTr/amos_0005.nii.gz,scan,Tr_5,MedImage,291404b6eca6b90014134fc09166a5b0c67d0191,"(768, 768, 80)",3,47185920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
3,Tr,6,amos22/imagesTr/amos_0006.nii.gz,scan,Tr_6,MedImage,a0dbfbc847622c27f8379106bd829d64bd77d77c,"(512, 512, 99)",3,25952256,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
4,Tr,7,amos22/imagesTr/amos_0007.nii.gz,scan,Tr_7,MedImage,ca96ae6008c490bdc46ab0f89474e7e03ce6217b,"(768, 768, 107)",3,63111168,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Va,573,amos22/labelsVa/amos_0573.nii.gz,mask,Va_573,Mask,f2333983065ac98b2a71b1457951aea05d3315db,"(320, 260, 72)",3,5990400,...,235.443000,0.420071,1.157307,1.728001,74.748770,70213.070716,"(113.01956759593764, 130.79836259630235, 226.0...",3.0,amos22/imagesVa/amos_0573.nii.gz,SEG
956,Va,575,amos22/labelsVa/amos_0575.nii.gz,mask,Va_575,Mask,3db4550bfed09c738dc7b18012dc2d7e0f455799,"(260, 144, 320)",3,11980800,...,366.223913,0.382038,1.387391,1.378596,90.089618,101990.413850,"(130.14465587173495, 180.56158393522423, 248.9...",3.0,amos22/imagesVa/amos_0575.nii.gz,SEG
957,Va,576,amos22/labelsVa/amos_0576.nii.gz,mask,Va_576,Mask,17f722934f527aa4b4d8548d7117eb6c7b85e1a8,"(320, 260, 72)",3,5990400,...,284.810525,0.399127,1.168343,1.550508,84.699301,90150.784847,"(131.93846318752645, 154.14944500519806, 239.0...",13.0,amos22/imagesVa/amos_0576.nii.gz,SEG
958,Va,581,amos22/labelsVa/amos_0581.nii.gz,mask,Va_581,Mask,56423d0ec2e8d532cacbfccf0ca743e2650b96a2,"(576, 468, 72)",3,19408896,...,284.968694,0.478873,1.336116,1.461998,85.326695,91491.281922,"(123.94713972021772, 165.60775457070784, 242.1...",2.0,amos22/imagesVa/amos_0581.nii.gz,SEG


In [20]:
def map_modality(row):
    if row["file_type"] == "scan":
        id = row["ImageID"]
        if id <= 500:
            return "CT"
        else:
            return "MR"
    elif row["file_type"] == "mask":
        return "SEG"

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df

,Split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,nvoxels,...,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality
0,Tr,1,amos22/imagesTr/amos_0001.nii.gz,scan,Tr_1,MedImage,7afee8a38d53124559a13a6781f122286d3430f7,"(768, 768, 90)",3,53084160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
1,Tr,4,amos22/imagesTr/amos_0004.nii.gz,scan,Tr_4,MedImage,c30d47e4286bcdfd0de0e7a91f2c83fdbf5f6c73,"(512, 512, 78)",3,20447232,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
2,Tr,5,amos22/imagesTr/amos_0005.nii.gz,scan,Tr_5,MedImage,291404b6eca6b90014134fc09166a5b0c67d0191,"(768, 768, 80)",3,47185920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
3,Tr,6,amos22/imagesTr/amos_0006.nii.gz,scan,Tr_6,MedImage,a0dbfbc847622c27f8379106bd829d64bd77d77c,"(512, 512, 99)",3,25952256,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
4,Tr,7,amos22/imagesTr/amos_0007.nii.gz,scan,Tr_7,MedImage,ca96ae6008c490bdc46ab0f89474e7e03ce6217b,"(768, 768, 107)",3,63111168,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Va,573,amos22/labelsVa/amos_0573.nii.gz,mask,Va_573,Mask,f2333983065ac98b2a71b1457951aea05d3315db,"(320, 260, 72)",3,5990400,...,235.443000,0.420071,1.157307,1.728001,74.748770,70213.070716,"(113.01956759593764, 130.79836259630235, 226.0...",3.0,amos22/imagesVa/amos_0573.nii.gz,SEG
956,Va,575,amos22/labelsVa/amos_0575.nii.gz,mask,Va_575,Mask,3db4550bfed09c738dc7b18012dc2d7e0f455799,"(260, 144, 320)",3,11980800,...,366.223913,0.382038,1.387391,1.378596,90.089618,101990.413850,"(130.14465587173495, 180.56158393522423, 248.9...",3.0,amos22/imagesVa/amos_0575.nii.gz,SEG
957,Va,576,amos22/labelsVa/amos_0576.nii.gz,mask,Va_576,Mask,17f722934f527aa4b4d8548d7117eb6c7b85e1a8,"(320, 260, 72)",3,5990400,...,284.810525,0.399127,1.168343,1.550508,84.699301,90150.784847,"(131.93846318752645, 154.14944500519806, 239.0...",13.0,amos22/imagesVa/amos_0576.nii.gz,SEG
958,Va,581,amos22/labelsVa/amos_0581.nii.gz,mask,Va_581,Mask,56423d0ec2e8d532cacbfccf0ca743e2650b96a2,"(576, 468, 72)",3,19408896,...,284.968694,0.478873,1.336116,1.461998,85.326695,91491.281922,"(123.94713972021772, 165.60775457070784, 242.1...",2.0,amos22/imagesVa/amos_0581.nii.gz,SEG


In [34]:
index_df["BodyPartExamined"] = "ABDOMEN"

In [36]:
import json

dataset_json = temp_data_path / "amos22" / "dataset.json"
with open(dataset_json, "r") as f:
    dataset = json.load(f)

dataset

{'name': 'AMOS',
 'description': 'Amos: A large-scale abdominal multi-organ benchmark for versatile medical image segmentation',
 'author': 'Yuanfeng Ji',
 'reference': 'SRIDB x CUHKSZ x HKU x LGCHSZ x LGPHSZ',
 'licence': 'CC-BY-SA 4.0',
 'release': '1.0 01/05/2022',
 'contact': 'u3008013@connect.hku.hk',
 'tensorImageSize': '3D',
 'modality': {'0': 'CT'},
 'labels': {'0': 'background',
  '1': 'spleen',
  '2': 'right kidney',
  '3': 'left kidney',
  '4': 'gall bladder',
  '5': 'esophagus',
  '6': 'liver',
  '7': 'stomach',
  '8': 'arota',
  '9': 'postcava',
  '10': 'pancreas',
  '11': 'right adrenal gland',
  '12': 'left adrenal gland',
  '13': 'duodenum',
  '14': 'bladder',
  '15': 'prostate/uterus'},
 'numTraining': 240,
 'numValidation': 120,
 'numTest': 240,
 'training': [{'image': './imagesTr/amos_0001.nii.gz',
   'label': './labelsTr/amos_0001.nii.gz'},
  {'image': './imagesTr/amos_0004.nii.gz',
   'label': './labelsTr/amos_0004.nii.gz'},
  {'image': './imagesTr/amos_0005.nii.gz

In [39]:
labels = dataset["labels"]
labels = ", ".join([v for k, v in labels.items() if v != "background"])
print(labels)


spleen, right kidney, left kidney, gall bladder, esophagus, liver, stomach, arota, postcava, pancreas, right 
adrenal gland, left adrenal gland, duodenum, bladder, prostate/uterus

In [41]:
def map_roinames(row):
    if row["file_type"] == "mask":
        return labels
    else:
        return ""

index_df["ROINames"] = index_df.apply(map_roinames, axis=1)
index_df


,Split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,nvoxels,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,BodyPartExamined,ROINames
0,Tr,1,amos22/imagesTr/amos_0001.nii.gz,scan,Tr_1,MedImage,7afee8a38d53124559a13a6781f122286d3430f7,"(768, 768, 90)",3,53084160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,
1,Tr,4,amos22/imagesTr/amos_0004.nii.gz,scan,Tr_4,MedImage,c30d47e4286bcdfd0de0e7a91f2c83fdbf5f6c73,"(512, 512, 78)",3,20447232,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,
2,Tr,5,amos22/imagesTr/amos_0005.nii.gz,scan,Tr_5,MedImage,291404b6eca6b90014134fc09166a5b0c67d0191,"(768, 768, 80)",3,47185920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,
3,Tr,6,amos22/imagesTr/amos_0006.nii.gz,scan,Tr_6,MedImage,a0dbfbc847622c27f8379106bd829d64bd77d77c,"(512, 512, 99)",3,25952256,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,
4,Tr,7,amos22/imagesTr/amos_0007.nii.gz,scan,Tr_7,MedImage,ca96ae6008c490bdc46ab0f89474e7e03ce6217b,"(768, 768, 107)",3,63111168,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Va,573,amos22/labelsVa/amos_0573.nii.gz,mask,Va_573,Mask,f2333983065ac98b2a71b1457951aea05d3315db,"(320, 260, 72)",3,5990400,...,1.157307,1.728001,74.748770,70213.070716,"(113.01956759593764, 130.79836259630235, 226.0...",3.0,amos22/imagesVa/amos_0573.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."
956,Va,575,amos22/labelsVa/amos_0575.nii.gz,mask,Va_575,Mask,3db4550bfed09c738dc7b18012dc2d7e0f455799,"(260, 144, 320)",3,11980800,...,1.387391,1.378596,90.089618,101990.413850,"(130.14465587173495, 180.56158393522423, 248.9...",3.0,amos22/imagesVa/amos_0575.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."
957,Va,576,amos22/labelsVa/amos_0576.nii.gz,mask,Va_576,Mask,17f722934f527aa4b4d8548d7117eb6c7b85e1a8,"(320, 260, 72)",3,5990400,...,1.168343,1.550508,84.699301,90150.784847,"(131.93846318752645, 154.14944500519806, 239.0...",13.0,amos22/imagesVa/amos_0576.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."
958,Va,581,amos22/labelsVa/amos_0581.nii.gz,mask,Va_581,Mask,56423d0ec2e8d532cacbfccf0ca743e2650b96a2,"(576, 468, 72)",3,19408896,...,1.336116,1.461998,85.326695,91491.281922,"(123.94713972021772, 165.60775457070784, 242.1...",2.0,amos22/imagesVa/amos_0581.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."


In [42]:
index_df.to_csv(indexed_datasets.imgtools_path / "AMOS" / "index.csv", index=False)

In [1]:
from imgnet.collections.store import IndexedDatasets

indexed_datasets = IndexedDatasets()

/home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
14:34:03 [info     ] Indexed datasets path: /home/gpudual/.local/share/med-imagenet/indexed_datasets [imgnet] call=store.__init__:67


In [2]:
from imgindex.model import validate_index

index = indexed_datasets.index("AMOS")
validate_index(index, "nifti", lazy=True)

Invalid NIFTI index: {
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": "NiftiIndex",
                "column": "NiftiIndex",
                "check": "column_in_dataframe",
                "error": "column 'SampleID' not in dataframe. Columns in dataframe: ['Split', 'ImageID', 'filepath', 'file_type', 'reference_id', 'class', 'hash', 'size', 'ndim', 'nvoxels', 'spacing', 'origin', 'direction', 'min', 'max', 'sum', 'mean', 'std', 'variance', 'dtype_str', 'dtype_numpy', 'mask.bbox.size', 'mask.bbox.min_coord', 'mask.bbox.max_coord', 'mask.feret_diameter', 'mask.roundness', 'mask.flatness', 'mask.elongation', 'mask.equivalent_spherical_radius', 'mask.equivalent_spherical_perimeter', 'mask.equivalent_ellipsoid_diameters', 'mask.volume_count', 'reference_scan', 'Modality', 'BodyPartExamined', 'ROINames']"
            }
        ]
    }
}


In [3]:
index.rename(columns={"reference_id": "SampleID"}, inplace=True)

In [4]:
validate_index(index, "nifti", lazy=True)

,Split,ImageID,filepath,file_type,SampleID,class,hash,size,ndim,nvoxels,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,BodyPartExamined,ROINames
0,Tr,1,amos22/imagesTr/amos_0001.nii.gz,scan,Tr_1,MedImage,7afee8a38d53124559a13a6781f122286d3430f7,"(768, 768, 90)",3,53084160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,NaN
1,Tr,4,amos22/imagesTr/amos_0004.nii.gz,scan,Tr_4,MedImage,c30d47e4286bcdfd0de0e7a91f2c83fdbf5f6c73,"(512, 512, 78)",3,20447232,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,NaN
2,Tr,5,amos22/imagesTr/amos_0005.nii.gz,scan,Tr_5,MedImage,291404b6eca6b90014134fc09166a5b0c67d0191,"(768, 768, 80)",3,47185920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,NaN
3,Tr,6,amos22/imagesTr/amos_0006.nii.gz,scan,Tr_6,MedImage,a0dbfbc847622c27f8379106bd829d64bd77d77c,"(512, 512, 99)",3,25952256,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,NaN
4,Tr,7,amos22/imagesTr/amos_0007.nii.gz,scan,Tr_7,MedImage,ca96ae6008c490bdc46ab0f89474e7e03ce6217b,"(768, 768, 107)",3,63111168,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,ABDOMEN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Va,573,amos22/labelsVa/amos_0573.nii.gz,mask,Va_573,Mask,f2333983065ac98b2a71b1457951aea05d3315db,"(320, 260, 72)",3,5990400,...,1.157307,1.728001,74.748770,70213.070716,"(113.01956759593764, 130.79836259630235, 226.0...",3.0,amos22/imagesVa/amos_0573.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."
956,Va,575,amos22/labelsVa/amos_0575.nii.gz,mask,Va_575,Mask,3db4550bfed09c738dc7b18012dc2d7e0f455799,"(260, 144, 320)",3,11980800,...,1.387391,1.378596,90.089618,101990.413850,"(130.14465587173495, 180.56158393522423, 248.9...",3.0,amos22/imagesVa/amos_0575.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."
957,Va,576,amos22/labelsVa/amos_0576.nii.gz,mask,Va_576,Mask,17f722934f527aa4b4d8548d7117eb6c7b85e1a8,"(320, 260, 72)",3,5990400,...,1.168343,1.550508,84.699301,90150.784847,"(131.93846318752645, 154.14944500519806, 239.0...",13.0,amos22/imagesVa/amos_0576.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."
958,Va,581,amos22/labelsVa/amos_0581.nii.gz,mask,Va_581,Mask,56423d0ec2e8d532cacbfccf0ca743e2650b96a2,"(576, 468, 72)",3,19408896,...,1.336116,1.461998,85.326695,91491.281922,"(123.94713972021772, 165.60775457070784, 242.1...",2.0,amos22/imagesVa/amos_0581.nii.gz,SEG,ABDOMEN,"spleen, right kidney, left kidney, gall bladde..."


In [5]:
from pathlib import Path

new_index_dir = Path("/home/gpudual/bhklab/josh/med-image_index/new_index")
collection_path = new_index_dir / "AMOS"
collection_path.mkdir(parents=True, exist_ok=True)

index.to_csv(collection_path / "index.csv", index=False)

In [6]:
import json
source = indexed_datasets.source_config("AMOS")

with open(collection_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)